[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AGRIdigital-Climbat/PlantillaMapa/blob/main/03_Temperatura/01_temperatura.ipynb)

# Temperatura del aire y del suelo: regimenes y tendencia
## PROSUCO - CLIMBAT | Analisis por municipio de Bolivia (GEE + Colab)

> **Area de estudio = tu GeoJSON.** Este notebook usa por defecto:
>
> `RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"` (compartido `Cualquiera con el enlace`).
> El GeoJSON **siempre manda**: al cargarse, `roi`, `ZONA['bbox']`, `ZONA['centro']`
> y `area_km2` se recalculan desde el poligono. El `bbox` de CONFIG solo es respaldo
> si cambias a `AOI_MODE = "bbox"`.

**Orden de ejecucion:** 1) Instalacion -> 2) Autenticacion GEE (`EE_PROJECT`) ->
3) CONFIG (anos + satelite + umbrales) -> 4) AOI desde Drive -> 5) celdas del modulo.
Solo edita la celda **CONFIGURACION MAESTRA**.


In [1]:
# ============================================================
# 1.1 Instalación de librerías (ejecutar una sola vez en Colab)
# ============================================================

import importlib
import subprocess
import sys


def _asegurar(paquete, modulo=None):
    """Instala el paquete solo si no está disponible."""
    try:
        importlib.import_module(modulo or paquete)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", paquete]
        )


_asegurar("earthengine-api", "ee")
_asegurar("geemap")
_asegurar("geopandas")
_asegurar("pandas")
_asegurar("numpy")
_asegurar("matplotlib")

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("Librerías listas.")
print("earthengine-api:", ee.__version__)
print("geemap:", geemap.__version__)

Librerías listas.
earthengine-api: 1.7.39
geemap: 0.38.8


In [2]:
# ============================================================
# 1.2 Autenticación en Google Earth Engine
# ============================================================
# >>> REEMPLAZAR por el id del proyecto de Cloud de PROSUCO/CLIMBAT <<<

EE_PROJECT = "gen-lang-client-0636031685"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine inicializado correctamente.")
print("Proyecto:", EE_PROJECT)

Earth Engine inicializado correctamente.
Proyecto: gen-lang-client-0636031685


## Configuración maestra

> **✏️ Esta es la única celda que se edita para diagnosticar una zona nueva.**

| Bloque | Qué define | Verificación previa |
|---|---|---|
| `ZONA` | nombre, bbox `[oeste, sur, este, norte]` y punto de vista | dibujar en [geojson.io](https://geojson.io) o en el editor GEE y copiar coordenadas |
| `AOI_MODE`/`RUTA_AOI` | fuente del AOI: `"bbox"` o `"auto"` (polígono **GeoJSON**) | exportar el polígono desde QGIS como **GeoJSON**; en `RUTA_AOI` se puede pegar un **enlace de Google Drive** (el notebook lo descarga solo) o una ruta de Drive |
| `PUNTOS_LOCALES` | manantiales/bofedales del saber local | levantamiento con las comunidades |
| `FECHA_INICIO / FECHA_FIN` | ventana analítica (**30+ años**) | default 1996-01-01 → 2026-12-31 |
| `CALENDARIO` | meses de siembra/cosecha y estación seca | observadores Yapuchiri |
| `UMBRALES` | todos los cortes analíticos en un solo lugar | criterio técnico + experiencia local |
| `ESCALAS` | resolución de cómputo (m); subir si la zona es grande | — |

In [3]:
# ===========================================================================
# CONFIGURACIÓN MAESTRA — ÚNICA CELDA QUE SE EDITA PARA CADA ZONA
# ===========================================================================

# --- 1. Zona de intervención ------------------------------------------------
# Fuente del AOI:
#   "bbox" -> usa el rectángulo ZONA["bbox"] de abajo (rápido, sin Drive)
#   "auto" -> usa un polígono .geojson. RUTA_AOI admite 3 formas:
#       * Enlace o ID de Google Drive  -> se DESCARGA solo (no subes nada a mano)
#           "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"
#           "11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N"
#       * Ruta en tu Drive ya montado  -> se lee directo, SIN copiar a /content
#           "/content/drive/MyDrive/CLIMBAT/zona.geojson"
#       * Carpeta en Drive             -> toma el único .geojson que contenga
#           "/content/drive/MyDrive/CLIMBAT"
AOI_MODE = "auto"
RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"

ZONA = {
    # Coordenadas geocodificadas: Escoma (La Paz, Bolivia) ~3832 msnm
    # (solo se usan como respaldo cuando AOI_MODE = "bbox")
    "nombre": "Escoma, La Paz (prueba)",
    "bbox": [-69.33, -15.86, -68.93, -15.46],   # [oeste, sur, este, norte]
    "centro": [-69.1289, -15.6615],             # punto de vista del mapa
    "zoom": 10,
}

# --- 2. Puntos levantados con comunidades (saber local) ---------------------
# Listas de [longitud, latitud]. Dejar vacías si aún no hay levantamiento.
PUNTOS_LOCALES = {
    "manantiales": [],
    "bofedales": [],
}

# --- 3. Ventana analítica: 30+ años hacia atrás desde el año actual ---------
# Regla de la plantilla: 1996-01-01 -> 2026-12-31 (todo análisis de rango de años usa esta).
# Los datasets usan su período nativo si es mayor: CHIRPS 1981+ · ERA5-Land 1950+
# · JRC 1984–2021 · MODIS 2000+ (NDVI) / 2001+ (MCD12Q1 agricultura) · SMAP 2015+
FECHA_INICIO = "1996-01-01"
FECHA_FIN = "2026-12-31"

# --- 4. Calendario agrícola local (verificar con observadores Yapuchiri) ----
CALENDARIO = {
    "siembra": 11,       # mes de siembra (1–12)
    "cosecha": 4,        # mes de cosecha (1–12)
    "meses_estacion_seca": [5, 6, 7, 8, 9, 10],   # se usa en M10 (bofedales)
}

# --- 5. Umbrales analíticos (ajustar con criterio técnico y local) ----------
UMBRALES = {
    "helada_c": 0.0,               # °C mínima para contar un día de helada (M4)
    "helada_grave_c": -2.0,        # °C de helada grave (daño severo) (M4)
    "mes_seco_mm": 30.0,           # mm/mes para clasificar mes seco (M2)
    "spi_severo": -1.5,            # umbral de sequía severa SPI-3 (M5)
    "ndvi_anomalia_z": -1.5,       # anomalía NDVI severa (M7)
    "manantial_occurrencia": 100,  # % ocurrencia JRC mínima (M9)
    "manantial_drenaje_max": 50,   # celdas acumuladas máx. (cabeceras) (M9)
    "bofedal_altitud_min": 3000,   # msnm mínimo (M10)
    "bofedal_altitud_max": 5200,   # msnm máximo (M10)
    "bofedal_pendiente_max": 5.0,  # grados de pendiente máxima (M10)
    "bofedal_seasonality": 8,      # meses con agua JRC mínimos (M10)
    "bofedal_ndvi_seco": 0.25,     # NDVI mínimo en estación seca (M10)
    "clases_agricolas": [12, 14],  # LC_Type1 MCD12Q1: 12=cultivos, 14=mosaico (M11)
}

# --- 6. Escalas de cómputo (metros) -----------------------------------------
# Subir si la zona es muy grande (más rápido, menos detalle)
ESCALAS = {"clima": 5000, "modis": 250, "jrc": 90, "detalle": 30}
TILE_SCALE = 4

# ===========================================================================
# A partir de aquí no se edita nada
# ===========================================================================

MESES = list(range(1, 13))
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
            "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

roi = None   # se construye en la celda 1.4 (bbox o GeoJSON según AOI_MODE)
centro = None
ANIO_INI = int(FECHA_INICIO[:4])
ANIO_FIN = int(FECHA_FIN[:4])

# Años COMPLETOS (se excluye el año en curso de métricas y mapas anuales)
ANIOS_COMPLETOS = list(range(ANIO_INI, ANIO_FIN))
FILTRO_ANIOS_COMPLETOS = ee.Filter.calendarRange(
    ANIOS_COMPLETOS[0], ANIOS_COMPLETOS[-1], "year"
)

print("Zona:", ZONA["nombre"])
print("Ventana analítica:", FECHA_INICIO, "->", FECHA_FIN,
      f"({ANIO_FIN - ANIO_INI + 1} años calendario)")
print("Años completos para métricas:", ANIOS_COMPLETOS[0], "-", ANIOS_COMPLETOS[-1])
print("Calendario: siembra mes", CALENDARIO["siembra"],
      "| cosecha mes", CALENDARIO["cosecha"])
print("Umbrales activos:", UMBRALES)

Zona: Escoma, La Paz (prueba)
Ventana analítica: 1996-01-01 -> 2026-12-31 (31 años calendario)
Años completos para métricas: 1996 - 2025
Calendario: siembra mes 11 | cosecha mes 4
Umbrales activos: {'helada_c': 0.0, 'helada_grave_c': -2.0, 'mes_seco_mm': 30.0, 'spi_severo': -1.5, 'ndvi_anomalia_z': -1.5, 'manantial_occurrencia': 100, 'manantial_drenaje_max': 50, 'bofedal_altitud_min': 3000, 'bofedal_altitud_max': 5200, 'bofedal_pendiente_max': 5.0, 'bofedal_seasonality': 8, 'bofedal_ndvi_seco': 0.25, 'clases_agricolas': [12, 14]}


In [4]:
# ============================================================
# 1.3 Utilidades comunes (reutilizadas por todos los módulos)
# ============================================================

# Parámetros de reducción seguros contra "User memory limit exceeded":
#   bestEffort=True -> GEE sube la escala automáticamente si no cabe en memoria.
#   tileScale=8     -> divide el cálculo en tiles más pequeños (menos memoria c/u).
REDUCIR = dict(maxPixels=1e9, tileScale=8, bestEffort=True)
REDUCIR_AREA = dict(maxPixels=1e10, tileScale=16, bestEffort=True)


def apilar(imagenes, nombres):
    """Une imágenes de una banda en una sola imagen multibanda."""
    pila = None
    for img, nombre in zip(imagenes, nombres):
        banda = ee.Image(img).select(0).rename(nombre)
        pila = banda if pila is None else pila.addBands(banda)
    return pila


def reducir_bandas(imagenes, nombres, escala, tam=6, reductor=None, params=None):
    """reduceRegion de muchas bandas EN BLOQUES (una petición por bloque).

    Evita el error de memoria que ocurre al reducir de una sola vez una imagen
    con decenas de bandas (cada banda puede depender de cientos/miles de imágenes
    diarias). Devuelve un dict {nombre_banda: valor}.
    """
    reductor = reductor or ee.Reducer.mean()
    params = params or REDUCIR
    salida = {}
    for i in range(0, len(imagenes), tam):
        pila = apilar(imagenes[i:i + tam], nombres[i:i + tam])
        salida.update(
            pila.reduceRegion(reductor, roi, escala, **params).getInfo()
        )
    return salida


def composite_mensual(coleccion, anios, banda=None):
    """Reduce una colección diaria a UN compuesto por año-mes (una imagen c/u).

    Aligera ~30x la memoria de las climatologías de 30 años. `anios` debe listar
    solo años con dato (evita colecciones vacías). Con `banda`, deja una sola banda.
    """
    imgs = []
    for y in anios:
        for m in MESES:
            i0 = f"{y}-{m:02d}-01"
            i1 = (pd.Timestamp(i0) + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")
            sub = coleccion.filterDate(i0, i1)
            if banda:
                sub = sub.select(banda)
            imgs.append(sub.mean().set("system:time_start", ee.Date(i0).millis()))
    return ee.ImageCollection(imgs)


def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente).

    Evita que el relleno de color tape el mapa y meta ruido en la interpretación;
    deja el límite del área bien visible por encima de las demás capas.
    """
    return roi.style(color=color, fillColor="00000000", width=ancho)


def promediar_aoi(imagen, banda, escala):
    """Promedio de una banda dentro del AOI -> valor Python (o None)."""
    d = (
        imagen.select(banda)
        .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        .getInfo()
    )
    return d.get(banda)


def serie_aoi(coleccion, banda, escala, bloque_anios=5):
    """Extrae el promedio del AOI por imagen, EN BLOQUES de años.

    Procesar la serie completa de una vez agota la memoria; por eso se trocea
    (por defecto cada 5 años) y se concatenan los resultados.
    """
    registros = []

    def _una_imagen(img):
        st = (
            img.select(banda)
            .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        )
        return ee.Feature(None, {
            "date": ee.Date(img.get("system:time_start")).format("yyyy-MM-dd"),
            "valor": st.get(banda),
        })

    for y0 in range(ANIO_INI, ANIO_FIN + 1, bloque_anios):
        y1 = min(y0 + bloque_anios - 1, ANIO_FIN)
        sub = coleccion.filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
        fc = ee.FeatureCollection(sub.map(_una_imagen)).getInfo()
        registros += [f["properties"] for f in fc["features"]]
    return registros


print("Utilidades: apilar(), reducir_bandas(), composite_mensual(), "
      "promediar_aoi(), serie_aoi()")

Utilidades: apilar(), reducir_bandas(), composite_mensual(), promediar_aoi(), serie_aoi()


In [5]:
# ============================================================
# 1.4 Área de estudio (AOI) y mapa base
# ============================================================
# Fuente del AOI (ver CONFIG):
#   AOI_MODE = "bbox" -> rectángulo ZONA["bbox"].
#   AOI_MODE = "auto" -> polígono desde un archivo .geojson en Google Drive.
#                        (En QGIS: Exportar -> Guardar entidades como -> GeoJSON,
#                         CRS EPSG:4326; subir a Drive y ajustar RUTA_AOI.)

def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente)."""
    # ee.Geometry NO tiene .style() en la API de Python -> se envuelve en FeatureCollection
    fc = ee.FeatureCollection([ee.Feature(roi)])
    return fc.style(color=color, fillColor="00000000", width=ancho)


if AOI_MODE == "auto":
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    import re
    import glob
    import geopandas as gpd

    ruta = str(RUTA_AOI).strip()

    # (a) Enlace o ID de Google Drive -> descarga automática (sin subir nada a mano).
    #     El archivo debe estar compartido como "Cualquiera con el enlace".
    if ruta.startswith("http") or ("/" not in ruta and len(ruta) >= 20):
        _asegurar("gdown")
        import gdown

        m = re.search(r"(?:/d/|id=)([-\w]{20,})", ruta)
        fid = m.group(1) if m else ruta
        ext = (
            ".zip" if ruta.lower().endswith(".zip") else
            ".kml" if ruta.lower().endswith(".kml") else
            ".gpkg" if ruta.lower().endswith(".gpkg") else
            ".geojson"
        )
        destino = "/content/zona_aoi" + ext
        print("Descargando AOI desde Google Drive...")
        gdown.download(id=fid, output=destino, quiet=False)
        ruta = destino
    # (b) Carpeta en Drive -> usar el único .geojson que contenga
    elif os.path.isdir(ruta):
        cands = sorted(glob.glob(os.path.join(ruta, "*.geojson")))
        if not cands:
            raise FileNotFoundError(f"No se encontró ningún .geojson en {ruta}")
        ruta = cands[0]
    # (c) Ruta de archivo en el Drive montado -> se lee directo (sin copiar)

    gdf = gpd.read_file(ruta).to_crs(4326)
    try:
        geometria = gdf.geometry.union_all()      # shapely >= 2 (disuelve features)
    except AttributeError:
        geometria = gdf.geometry.unary_union      # shapely < 2
    gdf = gpd.GeoDataFrame(geometry=[geometria], crs=4326)

    roi = geemap.geopandas_to_ee(gdf).geometry()

    minx, miny, maxx, maxy = geometria.bounds
    ZONA["bbox"] = [minx, miny, maxx, maxy]
    ZONA["centro"] = [(minx + maxx) / 2, (miny + maxy) / 2]
    ZONA["zoom"] = 10
    print("AOI cargado desde:", ruta)
else:
    roi = ee.Geometry.Rectangle(ZONA["bbox"], proj="EPSG:4326", geodesic=False)

centro = ee.Geometry.Point(ZONA["centro"])
area_km2 = roi.area(maxError=1e6).divide(1e6).getInfo()

Map_aoi = geemap.Map()
Map_aoi.centerObject(roi, ZONA["zoom"])
Map_aoi.add_basemap("HYBRID")
Map_aoi.addLayer(borde_aoi(), {}, f"AOI: {ZONA['nombre']}")

# Puntos del saber local (si se cargaron en CONFIG)
for tipo, color in (("manantiales", "cyan"), ("bofedales", "magenta")):
    coords = PUNTOS_LOCALES.get(tipo, [])
    if coords:
        Map_aoi.addLayer(
            ee.Geometry.MultiPoint(coords),
            {"color": color},
            f"{tipo.capitalize()} (saber local)",
        )

print(f"Zona: {ZONA['nombre']}")
print(f"Área del AOI: {area_km2:,.1f} km²")
print(f"Extensión (bbox): {ZONA['bbox']}")
print("Puntos de saber local cargados:",
      {k: len(v) for k, v in PUNTOS_LOCALES.items()})

Map_aoi

Mounted at /content/drive
Descargando AOI desde Google Drive...


Downloading...
From: https://drive.google.com/uc?id=11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N
To: /content/zona_aoi.geojson
100%|██████████| 6.32k/6.32k [00:00<00:00, 8.88MB/s]


AOI cargado desde: /content/zona_aoi.geojson
Zona: Escoma, La Paz (prueba)
Área del AOI: 5,638.9 km²
Extensión (bbox): [-68.088142, -19.88671, -67.170426, -19.050138]
Puntos de saber local cargados: {'manantiales': 0, 'bofedales': 0}


Map(center=[-19.436538457151876, -67.61413826547425], controls=(WidgetControl(options=['position', 'transparen…

# M3. Temperatura del aire: regímenes térmicos y tendencia
**Dimensiones:** *estacionalidad* (ciclo anual), *intensidad* (rangos extremos),
*tendencia* (°C/década, exploratoria).

* **Primario:** `ECMWF/ERA5_LAND/DAILY_AGGR` — reanálisis diario 1950–actual, ~11 km, con
  `temperature_2m_min` y `temperature_2m_max` en Kelvin (se convierte a °C).
* **Alternativas:** TerraClimate (mensual, 4,6 km, bandas `tmmx/tmmn` con **factor de escala ×0,1**)
  y LST MODIS (1 km, temperatura de *superficie*, no del aire: útil para detalle espacial pero
  con sesgo respecto al aire — crudo para píxeles con nieve/roca).
* **Nota CLIMBAT:** los valores son de rejilla (~11 km); el microclima de un valle o una ladera
  puede desviarse varios grados. Contrastar siempre con las estaciones locales.

In [8]:
# ============================================================
# Dos mapas lado a lado: T MÍNIMA y T MÁXIMA DEL AIRE por año
# (°C) — selector desplegable (refresco forzado)
# Fuente: ERA5-Land (temperatura del aire a 2 m)
# ============================================================

import ipywidgets as widgets
from IPython.display import display
import time
import ee

# --- Definir t2m_diario (ERA5-Land convertido a Celsius) ---
COLECCION = "ECMWF/ERA5_LAND/DAILY_AGGR"

def convertir_a_celsius(img):
    tmax = img.select("temperature_2m_max").subtract(273.15).rename("tmax")
    tmin = img.select("temperature_2m_min").subtract(273.15).rename("tmin")
    return tmax.addBands(tmin).copyProperties(img, ["system:time_start"])

t2m_diario = (
    ee.ImageCollection(COLECCION)
    .filterDate(FECHA_INICIO, FECHA_FIN)
    .map(convertir_a_celsius)
)

# --- Paleta en °C -----------------------------------------------------------
vis_t = {"min": -10, "max": 25,
         "palette": ["313695", "4575b4", "74add1", "fee090", "f46d43", "a50026"]}

# --- Imagen anual (temperatura del aire ERA5-Land) --------------------------
def _anual(anio, banda):
    """Media anual de la banda (tmin|tmax) del aire para un año dado."""
    return (t2m_diario
            .filter(ee.Filter.calendarRange(anio, anio, "year"))
            .select(banda)
            .mean()
            .rename(banda)
            .clip(roi))

# --- DIAGNÓSTICO: verificar que los datos cambian entre años ----------------
print("=" * 60)
print("VERIFICACIÓN NUMÉRICA DE LA TEMPERATURA DEL AIRE POR AÑO")
print("=" * 60)

for anio in [ANIOS_COMPLETOS[0],
             ANIOS_COMPLETOS[len(ANIOS_COMPLETOS)//2],
             ANIOS_COMPLETOS[-1]]:
    img_tmin = _anual(anio, "tmin")
    img_tmax = _anual(anio, "tmax")

    stats = img_tmin.addBands(img_tmax).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=10000,
        maxPixels=1e13,
        bestEffort=True,
    ).getInfo()

    print(f"Año {anio}:")
    print(f"  tmin media = {stats.get('tmin'):.3f} °C")
    print(f"  tmax media = {stats.get('tmax'):.3f} °C")
print("=" * 60)
print("Si los valores cambian entre años, los datos están bien.")
print("Si son idénticos, hay un bug en t2m_diario o en _anual.")
print("=" * 60 + "\n")

# --- Mapas base --------------------------------------------------------------
anio_default = ANIOS_COMPLETOS[-1]

def _mk_map():
    m = geemap.Map(layout=widgets.Layout(width="100%", height="560px"))
    m.centerObject(roi, ZONA["zoom"] - 1)
    m.add_basemap("HYBRID")
    m.addLayer(borde_aoi(), {}, "AOI")
    return m

Map_min = _mk_map()
Map_max = _mk_map()

PREFIJO = "TEMP_"   # marca las capas dinámicas para poder borrarlas

def _set_layer(m, img, vis, nombre):
    """Reemplaza la capa dinámica forzando refresh con nombre único."""
    # 1) Eliminar TODAS las capas dinámicas previas
    for capa in list(m.layers):
        if getattr(capa, "name", "").startswith(PREFIJO):
            try:
                m.remove_layer(capa)
            except Exception:
                pass
    # 2) Añadir la nueva capa con sufijo temporal único
    #    El sufijo hace que ipyleaflet la trate como nueva y repinte.
    nombre_unico = f"{PREFIJO}{nombre}__{int(time.time()*1000)}"
    m.addLayer(img, vis, nombre_unico)

# --- Callback del selector ---------------------------------------------------
def _actualizar_par(*_):
    anio = w_anio.value
    print(f"[slider] Actualizando mapas al año {anio}...")
    _set_layer(Map_min, _anual(anio, "tmin"), vis_t,
               f"T min aire {anio} (°C)")
    _set_layer(Map_max, _anual(anio, "tmax"), vis_t,
               f"T max aire {anio} (°C)")

# --- Selector desplegable de año ---------------------------------------------
w_anio = widgets.Dropdown(
    options=list(ANIOS_COMPLETOS),
    value=anio_default,
    description="Año:",
    layout=widgets.Layout(width="200px"),
    style={"description_width": "50px"},
)
w_anio.observe(_actualizar_par, names="value")

# Primer render (agrega capas iniciales)
_actualizar_par()

# --- Encabezados con año dinámico --------------------------------------------
titulo_izq = widgets.HTML()
titulo_der = widgets.HTML()

def _actualizar_titulos(change):
    y = change["new"]
    titulo_izq.value = (f"<h4 style='text-align:center;margin:2px 0;'>"
                        f"T mínima del aire anual (°C) — {y}</h4>")
    titulo_der.value = (f"<h4 style='text-align:center;margin:2px 0;'>"
                        f"T máxima del aire anual (°C) — {y}</h4>")
w_anio.observe(_actualizar_titulos, names="value")
_actualizar_titulos({"new": anio_default})

# --- Layout 50/50 ------------------------------------------------------------
col_izq = widgets.VBox([titulo_izq, Map_min],
                       layout=widgets.Layout(width="50%", padding="0", margin="0"))
col_der = widgets.VBox([titulo_der, Map_max],
                       layout=widgets.Layout(width="50%", padding="0", margin="0"))

fila = widgets.HBox(
    [col_izq, col_der],
    layout=widgets.Layout(width="100%", padding="0", margin="0",
                          justify_content="space-between",
                          align_items="stretch"),
)

# --- Caja del selector (arriba del todo) -------------------------------------
caja_selector = widgets.HBox(
    [
        widgets.HTML("<b>Selecciona el año:</b>"),
        w_anio,
    ],
    layout=widgets.Layout(
        padding="10px",
        border="1px solid #ccc",
        width="fit-content",
        margin="0 0 8px 0",
    ),
)

display(widgets.VBox([
    widgets.HTML("<b>Temperatura del aire por año (ERA5-Land)</b> — "
                 "usa el selector para ver la T mínima (izq.) y T máxima (der.) "
                 "del aire a 2 m en cada año."),
    caja_selector,
    fila,
]))

VERIFICACIÓN NUMÉRICA DE LA TEMPERATURA DEL AIRE POR AÑO
Año 1996:
  tmin media = 1.501 °C
  tmax media = 16.405 °C
Año 2011:
  tmin media = 1.628 °C
  tmax media = 16.021 °C
Año 2025:
  tmin media = 1.938 °C
  tmax media = 16.531 °C
Si los valores cambian entre años, los datos están bien.
Si son idénticos, hay un bug en t2m_diario o en _anual.

[slider] Actualizando mapas al año 2025...


[slider] Actualizando mapas al año 2006...


In [ ]:
# ============================================================
# M3.2 Serie temporal anual de T mínima y T máxima del aire
# Dos gráficas apiladas: arriba T máxima, abajo T mínima
# Fuente: ERA5-Land diario (temperature_2m_max/min, ya en °C)
# ============================================================

anio_ini = ANIOS_COMPLETOS[0]
anio_fin = ANIOS_COMPLETOS[-1]
print(f"Serie anual: {anio_ini}-{anio_fin} ({len(ANIOS_COMPLETOS)} años)")

# --- Reductor por bloques con claves únicas --------------------------
def _reducir_chunks_1banda(anios, banda, escala, chunk=5):
    """Media anual de `banda` (tmin|tmax) del aire, por bloques."""
    vals = []
    n = len(anios)
    for i in range(0, n, chunk):
        sub = anios[i:i + chunk]
        imgs = [
            t2m_diario
            .filter(ee.Filter.calendarRange(y, y, "year"))
            .select(banda)
            .mean()
            .rename(banda)
            .set("system:index", f"y{y}")
            for y in sub
        ]
        img = ee.ImageCollection(imgs).toBands()
        stats = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=roi,
            scale=escala,
            maxPixels=1e13,
            bestEffort=True,
            tileScale=8,
        ).getInfo()
        if i == 0:
            print(f"  · claves toBands (muestra): {list(stats.keys())[:6]}")
        for y in sub:
            vals.append(stats.get(f"y{y}_{banda}", np.nan))
        print(f"  · {banda}: {min(i+chunk, n)}/{n} años reducidos")
    return vals

# --- Series anuales --------------------------------------------------
tmin_vals = _reducir_chunks_1banda(ANIOS_COMPLETOS, "tmin", ESCALAS["clima"], chunk=5)
tmax_vals = _reducir_chunks_1banda(ANIOS_COMPLETOS, "tmax", ESCALAS["clima"], chunk=5)

df_tserie = pd.DataFrame({
    "anio": ANIOS_COMPLETOS,
    "tmin": tmin_vals,
    "tmax": tmax_vals,
})

print("\nValores crudos (primeros 5 años):")
print(df_tserie.head().round(2))
print("NaNs -> tmin:", df_tserie["tmin"].isna().sum(),
      "| tmax:", df_tserie["tmax"].isna().sum())

if df_tserie[["tmin", "tmax"]].isna().all().all():
    raise RuntimeError("Todas las series son NaN. Revisa `t2m_diario`.")

display(df_tserie.round(2))

# ============================================================
# Dos gráficas apiladas (arriba máxima, abajo mínima)
# ============================================================

fig, axes = plt.subplots(
    2, 1,
    figsize=(13, 9),
    sharex=True,
    gridspec_kw={"hspace": 0.15},
)

# ---------- Gráfica superior: T MÁXIMA ANUAL ----------
ax1 = axes[0]
ax1.plot(df_tserie["anio"], df_tserie["tmax"],
         color="#d32f2f", lw=2, marker="o", ms=4, label="T máxima anual")
ax1.fill_between(df_tserie["anio"],
                 df_tserie["tmax"].min(),
                 df_tserie["tmax"],
                 color="#d32f2f", alpha=0.08)

m1 = df_tserie[["anio", "tmax"]].dropna()
if len(m1) >= 2:
    coef1 = np.polyfit(m1["anio"], m1["tmax"], 1)
    ax1.plot(m1["anio"], np.poly1d(coef1)(m1["anio"]),
             color="#d32f2f", ls="--", lw=1.3,
             label=f"Tendencia ({coef1[0]*10:+.2f} °C/década)")

media_tmax = df_tserie["tmax"].mean()
ax1.axhline(media_tmax, color="#d32f2f", ls=":", alpha=0.6,
            label=f"Media {media_tmax:.2f} °C")

ax1.set_title("Temperatura máxima del aire anual (°C)", fontsize=12)
ax1.set_ylabel("°C")
ax1.grid(alpha=0.3)
ax1.legend(loc="best", fontsize=9)

# ---------- Gráfica inferior: T MÍNIMA ANUAL ----------
ax2 = axes[1]
ax2.plot(df_tserie["anio"], df_tserie["tmin"],
         color="#1976d2", lw=2, marker="o", ms=4, label="T mínima anual")
ax2.fill_between(df_tserie["anio"],
                 df_tserie["tmin"],
                 df_tserie["tmin"].max(),
                 color="#1976d2", alpha=0.08)

m2 = df_tserie[["anio", "tmin"]].dropna()
if len(m2) >= 2:
    coef2 = np.polyfit(m2["anio"], m2["tmin"], 1)
    ax2.plot(m2["anio"], np.poly1d(coef2)(m2["anio"]),
             color="#1976d2", ls="--", lw=1.3,
             label=f"Tendencia ({coef2[0]*10:+.2f} °C/década)")

media_tmin = df_tserie["tmin"].mean()
ax2.axhline(media_tmin, color="#1976d2", ls=":", alpha=0.6,
            label=f"Media {media_tmin:.2f} °C")

ax2.set_title("Temperatura mínima del aire anual (°C)", fontsize=12)
ax2.set_xlabel("Año")
ax2.set_ylabel("°C")
ax2.grid(alpha=0.3)
ax2.legend(loc="best", fontsize=9)

fig.suptitle(f"Serie anual de temperaturas del aire ({anio_ini}-{anio_fin}) — ERA5-Land",
             fontsize=13, y=0.995)
plt.tight_layout()
plt.show()

# ---------- Resumen numérico ----------
print(f"T máxima media: {media_tmax:.2f} °C | Año más cálido: "
      f"{int(df_tserie.loc[df_tserie['tmax'].idxmax(), 'anio'])} "
      f"({df_tserie['tmax'].max():.2f} °C)")
print(f"T mínima media: {media_tmin:.2f} °C | Año más frío: "
      f"{int(df_tserie.loc[df_tserie['tmin'].idxmin(), 'anio'])} "
      f"({df_tserie['tmin'].min():.2f} °C)")

In [ ]:
# ============================================================
# Serie diaria + mensual + anual + TENDENCIA — TODO EN COLAB
# Fuente: ERA5-Land Daily Aggregated (temperatura a 2 m)
# Rango: intersección entre FECHA_INICIO/FECHA_FIN y la disponibilidad real
# ============================================================

%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ee

# --- Punto de referencia -----------------------------------------------------
try:
    punto = ee.Geometry.Point([LON, LAT])
    print(f"Reduciendo sobre punto LON={LON}, LAT={LAT}")
except NameError:
    punto = roi.centroid(maxError=1000)
    print("Reduciendo sobre el centroide del AOI")

# --- 0) Detectar el rango REAL de datos disponibles --------------------------
COLECCION = "ECMWF/ERA5_LAND/DAILY_AGGR"

# Fecha de inicio real de la colección
info_col = ee.ImageCollection(COLECCION).limit(1).first().getInfo()
fecha_primer_dato = pd.to_datetime(info_col["properties"]["system:time_start"], unit="ms")
print(f"Primer dato disponible en {COLECCION}: {fecha_primer_dato.date()}")

# Fecha de fin real
info_col_fin = ee.ImageCollection(COLECCION).sort("system:time_start", False).limit(1).first().getInfo()
fecha_ultimo_dato = pd.to_datetime(info_col_fin["properties"]["system:time_start"], unit="ms")
print(f"Último dato disponible en {COLECCION}: {fecha_ultimo_dato.date()}")

# Intersección con la ventana configurada en la maestra
fecha_ini_dt = max(pd.to_datetime(FECHA_INICIO), fecha_primer_dato)
fecha_fin_dt = min(pd.to_datetime(FECHA_FIN), fecha_ultimo_dato)

fecha_ini_era5 = fecha_ini_dt.strftime("%Y-%m-%d")
fecha_fin_era5 = fecha_fin_dt.strftime("%Y-%m-%d")

# Años completos que caen dentro de la intersección
anio_ini_efectivo = fecha_ini_dt.year if fecha_ini_dt.month == 1 and fecha_ini_dt.day == 1 else fecha_ini_dt.year + 1
anio_fin_efectivo = fecha_fin_dt.year if fecha_fin_dt.month == 12 and fecha_fin_dt.day == 31 else fecha_fin_dt.year - 1

ANIOS_ERA5 = [y for y in ANIOS_COMPLETOS
              if anio_ini_efectivo <= y <= anio_fin_efectivo]

print(f"\nVentana solicitada en la maestra : {FECHA_INICIO} → {FECHA_FIN}")
print(f"Ventana efectiva (intersección) : {fecha_ini_era5} → {fecha_fin_era5}")
print(f"Años completos a analizar       : {ANIOS_ERA5[0]}-{ANIOS_ERA5[-1]} "
      f"({len(ANIOS_ERA5)} años)")

if not ANIOS_ERA5:
    raise ValueError("No hay años completos dentro de la intersección. "
                     "Revisa FECHA_INICIO/FECHA_FIN.")

# --- 1) Colección DIARIA en °C (ERA5-Land) ----------------------------------
def convertir_a_celsius(img):
    tmax = (img.select("temperature_2m_max")
            .subtract(273.15)
            .rename("tmax"))
    tmin = (img.select("temperature_2m_min")
            .subtract(273.15)
            .rename("tmin"))
    return (tmax.addBands(tmin)
            .copyProperties(img, ["system:time_start"]))

t2m_diario = (
    ee.ImageCollection(COLECCION)
    .filterDate(fecha_ini_era5, fecha_fin_era5)
    .filterBounds(punto)
    .map(convertir_a_celsius)
)

n_imgs = t2m_diario.size().getInfo()
print("Imágenes diarias:", n_imgs)
if n_imgs == 0:
    raise RuntimeError("La colección ERA5-Land está vacía. Revisa fechas y AOI.")

# --- 2) Extraer la serie diaria por BLOQUES de años con getRegion ------------
def _get_region_bloques(coleccion, punto, anios, chunk_anios=5, escala=10000):
    """Devuelve un DataFrame con una fila por día usando getRegion()."""
    filas = []
    for i in range(0, len(anios), chunk_anios):
        bloque = anios[i:i + chunk_anios]
        col = coleccion.filter(ee.Filter.calendarRange(
            bloque[0], bloque[-1], "year"))

        n_bloque = col.size().getInfo()
        if n_bloque == 0:
            print(f"  · bloque {bloque[0]}-{bloque[-1]}: VACÍO, se omite")
            continue

        region = col.getRegion(punto, scale=escala).getInfo()
        cabecera = region[0]
        for fila in region[1:]:
            filas.append(dict(zip(cabecera, fila)))
        print(f"  · bloque {bloque[0]}-{bloque[-1]}: "
              f"{len(region) - 1} días extraídos")
    return pd.DataFrame(filas)

df_diaria = _get_region_bloques(t2m_diario, punto, ANIOS_ERA5,
                                chunk_anios=5, escala=10000)

if df_diaria.empty:
    raise RuntimeError("No se extrajo ningún dato diario. Revisa el AOI o las fechas.")

# Normalizar columnas y fecha
df_diaria["date"] = pd.to_datetime(df_diaria["time"], unit="ms")
df_diaria = df_diaria[["date", "tmax", "tmin"]].dropna().sort_values("date")
df_diaria["tmax"] = df_diaria["tmax"].astype(float)
df_diaria["tmin"] = df_diaria["tmin"].astype(float)
df_diaria["anio"] = df_diaria["date"].dt.year

# Recorte final: solo años dentro de ANIOS_ERA5
df_diaria = df_diaria[df_diaria["anio"].isin(ANIOS_ERA5)].copy()

print(f"\nDías totales extraídos: {len(df_diaria)}")
print(f"Años presentes: {sorted(df_diaria['anio'].unique())}")
display(df_diaria.head())

# --- Verificación de integridad ----------------------------------------------
anios_presentes = set(df_diaria["anio"].unique())
anios_esperados = set(ANIOS_ERA5)
anios_faltantes = sorted(anios_esperados - anios_presentes)
if anios_faltantes:
    print(f"⚠ Años sin datos: {anios_faltantes}")
else:
    print(f"✓ Todos los años {ANIOS_ERA5[0]}-{ANIOS_ERA5[-1]} tienen datos.")

# --- 3) Serie MENSUAL --------------------------------------------------------
df_mensual = (
    df_diaria
    .set_index("date")
    .resample("MS")
    .agg({"tmax": "median", "tmin": "median"})
    .reset_index()
    .dropna()
)

print(f"\nMeses totales: {len(df_mensual)}")

# --- 3b) Serie ANUAL y TENDENCIAS -------------------------------------------
df_anual = (
    df_diaria
    .groupby("anio")
    .agg({"tmax": "mean", "tmin": "mean"})
    .reset_index()
)

coef_tmax = np.polyfit(df_anual["anio"], df_anual["tmax"], 1)
coef_tmin = np.polyfit(df_anual["anio"], df_anual["tmin"], 1)
tend_tmax_dec = coef_tmax[0] * 10
tend_tmin_dec = coef_tmin[0] * 10

print(f"\nTendencia T máxima del aire: {tend_tmax_dec:+.2f} °C/década")
print(f"Tendencia T mínima del aire: {tend_tmin_dec:+.2f} °C/década")

# --- 3c) Media móvil de 12 meses ---------------------------------------------
df_mensual["tmax_mm12"] = df_mensual["tmax"].rolling(12, center=True).mean()
df_mensual["tmin_mm12"] = df_mensual["tmin"].rolling(12, center=True).mean()

# --- 3d) Downsampling del scatter diario -------------------------------------
PASO_SCATTER = 3
df_scatter = df_diaria.iloc[::PASO_SCATTER]
print(f"Scatter diario: {len(df_scatter)} puntos (de {len(df_diaria)} días)")

# --- 4) Gráfica --------------------------------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

ax1 = axes[0]
ax1.scatter(df_scatter["date"], df_scatter["tmax"],
            color="#d32f2f", s=6, alpha=0.10, label="Observaciones diarias")
ax1.plot(df_mensual["date"], df_mensual["tmax"],
         color="#d32f2f", lw=1, alpha=0.4, label="Mediana mensual")
ax1.plot(df_mensual["date"], df_mensual["tmax_mm12"],
         color="#b71c1c", lw=2.5, label="Media móvil 12 meses")
ax1.plot(df_anual["anio"], df_anual["tmax"],
         color="#000000", marker="o", ms=4, lw=1.2, alpha=0.8,
         label="Media anual")
ax1.plot(df_anual["anio"], np.poly1d(coef_tmax)(df_anual["anio"]),
         color="#000000", ls="--", lw=1.8,
         label=f"Tendencia ({tend_tmax_dec:+.2f} °C/década)")
ax1.set_ylabel("T máx aire (°C)")
ax1.set_title("Temperatura máxima del aire — tendencia ERA5-Land")
ax1.grid(alpha=0.3)
ax1.legend(loc="best", fontsize=8, ncol=2)

ax2 = axes[1]
ax2.scatter(df_scatter["date"], df_scatter["tmin"],
            color="#1565c0", s=6, alpha=0.10, label="Observaciones diarias")
ax2.plot(df_mensual["date"], df_mensual["tmin"],
         color="#1565c0", lw=1, alpha=0.4, label="Mediana mensual")
ax2.plot(df_mensual["date"], df_mensual["tmin_mm12"],
         color="#0d47a1", lw=2.5, label="Media móvil 12 meses")
ax2.plot(df_anual["anio"], df_anual["tmin"],
         color="#000000", marker="o", ms=4, lw=1.2, alpha=0.8,
         label="Media anual")
ax2.plot(df_anual["anio"], np.poly1d(coef_tmin)(df_anual["anio"]),
         color="#000000", ls="--", lw=1.8,
         label=f"Tendencia ({tend_tmin_dec:+.2f} °C/década)")
ax2.set_ylabel("T mín aire (°C)")
ax2.set_xlabel("Fecha")
ax2.set_title("Temperatura mínima del aire — tendencia ERA5-Land")
ax2.grid(alpha=0.3)
ax2.legend(loc="best", fontsize=8, ncol=2)

# >>> CORRECCIÓN: Fijar el eje X al rango real de los datos <<<
ax2.set_xlim(df_diaria["date"].min(), df_diaria["date"].max())

fig.suptitle(
    f"Temperatura del aire a 2 m ERA5-Land — {ANIOS_ERA5[0]}-{ANIOS_ERA5[-1]}\n"
    f"Diario + mensual + anual + tendencia lineal",
    fontsize=14,
)
plt.tight_layout()
plt.show()

# --- 5) Resumen --------------------------------------------------------------
print("\n" + "=" * 60)
print("RESUMEN DE TENDENCIAS — TEMPERATURA DEL AIRE (ERA5-Land)")
print("=" * 60)
print(f"Ventana solicitada : {FECHA_INICIO} → {FECHA_FIN}")
print(f"Ventana efectiva   : {fecha_ini_era5} → {fecha_fin_era5}")
print(f"Periodo analizado  : {ANIOS_ERA5[0]}-{ANIOS_ERA5[-1]} ({len(ANIOS_ERA5)} años)")
print(f"Años con datos     : {len(df_anual)}")
print(f"T máx media        : {df_anual['tmax'].mean():.2f} °C")
print(f"T mín media        : {df_anual['tmin'].mean():.2f} °C")
print("-" * 60)
print(f"Tendencia T máx: {tend_tmax_dec:+.2f} °C/década")
print(f"Tendencia T mín: {tend_tmin_dec:+.2f} °C/década")
print("=" * 60)

fig

In [ ]:
# ============================================================
# Serie diaria + mensual de LST — TODO EN COLAB Temperatura del suelo
# Fuente: MODIS Aqua (MYD11A1) — LST diurna (tmax) y nocturna (tmin)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ee

# --- Punto de referencia -----------------------------------------------------
try:
    punto = ee.Geometry.Point([LON, LAT])
    print(f"Reduciendo sobre punto LON={LON}, LAT={LAT}")
except NameError:
    punto = roi.centroid(maxError=1000)
    print("Reduciendo sobre el centroide del AOI")

# --- 0) Asegurar que el rango de años es compatible con MODIS ----------------
# MYD11A1 empieza el 2002-07-04. Si ANIOS_COMPLETOS incluye años anteriores,
# los recortamos automáticamente para evitar colecciones vacías.
ANIO_MODIS_INICIO = 2003   # primer año completo con datos MODIS
ANIOS_MODIS = [y for y in ANIOS_COMPLETOS if y >= ANIO_MODIS_INICIO]

if not ANIOS_MODIS:
    raise ValueError(
        f"MODIS MYD11A1 no tiene datos para los años {ANIOS_COMPLETOS}. "
        f"Empieza en {ANIO_MODIS_INICIO}."
    )

if len(ANIOS_MODIS) < len(ANIOS_COMPLETOS):
    print(f"⚠ Años recortados a MODIS: {ANIOS_MODIS[0]}-{ANIOS_MODIS[-1]} "
          f"({len(ANIOS_MODIS)} años)")

# Rango de fechas coherente con los años disponibles
fecha_ini_modis = f"{ANIOS_MODIS[0]}-01-01"
fecha_fin_modis = f"{ANIOS_MODIS[-1]}-12-31"

# --- 1) Colección DIARIA en °C (MODIS Aqua MYD11A1) -------------------------
def convertir_a_celsius(img):
    lst_dia = (img.select("LST_Day_1km")
               .multiply(0.02)
               .subtract(273.15)
               .rename("tmax"))
    lst_noche = (img.select("LST_Night_1km")
                 .multiply(0.02)
                 .subtract(273.15)
                 .rename("tmin"))
    return (lst_dia.addBands(lst_noche)
            .copyProperties(img, ["system:time_start"]))

t2m_diario = (
    ee.ImageCollection("MODIS/061/MYD11A1")
    .filterDate(fecha_ini_modis, fecha_fin_modis)
    .filterBounds(punto)
    .map(convertir_a_celsius)
)
n_imgs = t2m_diario.size().getInfo()
print("Imágenes diarias:", n_imgs)
if n_imgs == 0:
    raise RuntimeError("La colección MODIS está vacía. Revisa fechas y AOI.")

# --- 2) Extraer la serie diaria por BLOQUES de años con getRegion ------------
def _get_region_bloques(coleccion, punto, anios, chunk_anios=5, escala=1000):
    """Devuelve un DataFrame con una fila por día usando getRegion()."""
    filas = []
    for i in range(0, len(anios), chunk_anios):
        bloque = anios[i:i + chunk_anios]
        col = coleccion.filter(ee.Filter.calendarRange(
            bloque[0], bloque[-1], "year"))

        # Verificación: si el bloque está vacío, lo saltamos con aviso
        n_bloque = col.size().getInfo()
        if n_bloque == 0:
            print(f"  · bloque {bloque[0]}-{bloque[-1]}: VACÍO, se omite")
            continue

        region = col.getRegion(punto, scale=escala).getInfo()
        cabecera = region[0]
        for fila in region[1:]:
            filas.append(dict(zip(cabecera, fila)))
        print(f"  · bloque {bloque[0]}-{bloque[-1]}: "
              f"{len(region) - 1} días extraídos")
    return pd.DataFrame(filas)

df_diaria = _get_region_bloques(t2m_diario, punto, ANIOS_MODIS,
                                chunk_anios=5, escala=1000)

if df_diaria.empty:
    raise RuntimeError("No se extrajo ningún dato diario. Revisa el AOI o las fechas.")

# Normalizar columnas y fecha
df_diaria["date"] = pd.to_datetime(df_diaria["time"], unit="ms")
df_diaria = df_diaria[["date", "tmax", "tmin"]].dropna().sort_values("date")
df_diaria["tmax"] = df_diaria["tmax"].astype(float)
df_diaria["tmin"] = df_diaria["tmin"].astype(float)

print(f"\nDías totales extraídos: {len(df_diaria)}")
display(df_diaria.head())

# --- 3) Serie MENSUAL calculada en pandas ------------------------------------
df_mensual = (
    df_diaria
    .set_index("date")
    .resample("MS")
    .agg({"tmax": "median", "tmin": "median"})
    .reset_index()
    .dropna()
)

print(f"\nMeses totales: {len(df_mensual)}")
display(df_mensual.head())

# --- 4) Gráfica (2 paneles apilados) -----------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].scatter(df_diaria["date"], df_diaria["tmax"],
                color="#d32f2f", s=8, alpha=0.15,
                label="Observaciones diarias")
axes[0].plot(df_mensual["date"], df_mensual["tmax"],
             color="#d32f2f", marker="o", markersize=4, linewidth=2,
             label="Mediana mensual")
axes[0].set_ylabel("LST diurna (°C)")
axes[0].set_title("Variabilidad temporal de la LST diurna (MODIS Aqua)")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].scatter(df_diaria["date"], df_diaria["tmin"],
                color="#1565c0", s=8, alpha=0.15,
                label="Observaciones diarias")
axes[1].plot(df_mensual["date"], df_mensual["tmin"],
             color="#1565c0", marker="o", markersize=4, linewidth=2,
             label="Mediana mensual")
axes[1].set_ylabel("LST nocturna (°C)")
axes[1].set_xlabel("Fecha")
axes[1].set_title("Variabilidad temporal de la LST nocturna (MODIS Aqua)")
axes[1].grid(alpha=0.3)
axes[1].legend()

fig.suptitle(
    f"Temperatura de superficie MODIS Aqua (MYD11A1) — "
    f"{ANIOS_MODIS[0]}-{ANIOS_MODIS[-1]}\n"
    f"Observaciones diarias vs medianas mensuales",
    fontsize=14,
)
plt.tight_layout()
plt.show()

## Interpretación de M3
* **Rango térmico** (máx − mín): alto rango = estrés para cultivos y mayor depredador/eficiencia
  de riego; en el Altiplano el rango diario suele superar los 15–20 °C.
* **Mínima media anual < 0 °C:** marca zonas con frío de base (pisos altos) — ahí la helada no es
  evento sino condición.
* **Tendencia:** un valor > +0,2 °C/década es consistente con el calentamiento regional reportado
  para los Andes tropicales; verificar con Mann-Kendall antes de comunicarlo como tendencia.

**Preguntas guía**
1. ¿Los valles (inversiones térmicas) aparecen más fríos que las laderas en el mapa de mínimas?
2. ¿El rango térmico coincide con lo que miden sus estaciones CLIMBAT?
3. ¿Qué meses muestran las mínimas más bajas respecto a la siembra/cosecha del calendario?

## Cierre: checklist del modulo

- [ ] Parte base ejecutada (instalacion, autenticacion GEE, CONFIG, AOI del GeoJSON).
- [ ] Mapa(s) revisados contra el borde rojo del AOI (el GeoJSON siempre manda).
- [ ] Serie/tabla leidas + interpretacion contrastada con saber local (Yapuchiri/PACHASOL).
- [ ] Periodo y dataset anotados junto a cada numero (p. ej. `CHIRPS 1981-2025`).

### Como cambiar de satelite o de anos (patron en todos los modulos)

```python
FECHA_INICIO = "1996-01-01"  # en la celda CONFIG: ventana analitica
FECHA_FIN = "2026-12-31"
# --- Primario (default) ---
precip = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
# --- Alternativas: comentar la primaria y descomentar la elegida ---
# precip = ee.ImageCollection("NASA/GPM_L3/IMERG_V06")
# Cada alternativa indica resolucion, periodo y para que sirve.
```
